In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import tensorflow as keras
from keras import layers, Input, Model, ops
from keras.layers import Embedding, Dense
import tensorflow as tf
from collections import defaultdict
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import KLDivergence
import json


In [5]:
def HaversineMiles(coord1, coord2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1 = map(radians, coord1)
    lat2, lon2 = map(radians, coord2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    return R * c


def BuildEdgeIDHash(self):
    return dict(zip(self['edge_id'], zip(self['u'], self['v'])))

def BuildNodeToEdgesHash(self):
    node_to_edges = defaultdict(list)

    for _, row in self.iterrows():
        u = row['u']
        edge_id = row['edge_id']
        node_to_edges[u].append(edge_id)
    return node_to_edges

def BuildStaticFeatures(self):
    return self[['Distance_miles', 'TimeToCross_minutes', 'lon_u',
                        'lat_u', 'isHighPriority_u', 'lon_v', 'lat_v', 
                        'isHighPriority_v','heading_u', 'heading_v']].values

def build_valid_edge_ids_batch(agent_node_ids_row, NodeToEdgesHash):
    valid_edge_ids_batch = []

    for node_id in agent_node_ids_row:
        edge_ids = NodeToEdgesHash[node_id]  # List of ints
        edge_tensor = tf.convert_to_tensor(edge_ids, dtype=tf.int32)
        valid_edge_ids_batch.append(edge_tensor)

    return valid_edge_ids_batch

class PolicyHeader(tf.keras.Model):
    def __init__(self, num_edges, embed_dim):
        super().__init__()
        self.dense1 = Dense(128, activation='relu')
        self.dense2 = Dense(128, activation='relu')
        self.agent_proj = Dense(embed_dim)

        self.edge_embeddings = Embedding(num_edges, embed_dim)
        self.static_proj = Dense(embed_dim)

    def call(self, local_obs_batch, valid_edge_ids_batch, edge_static_features):
        if len(local_obs_batch.shape) == 3:
            local_obs_batch = tf.squeeze(local_obs_batch, axis=1)
        #print(local_obs_batch.shape)
        x = self.dense1(local_obs_batch)    
        x = self.dense2(x)                   
        agent_features = self.agent_proj(x)  

        all_probs = []

        batch_size = tf.shape(agent_features)[0]

        for i in range(5):
            ids = valid_edge_ids_batch[i] 

            valid_edge_vecs = self.edge_embeddings(ids)  
            valid_edge_static = tf.gather(edge_static_features, ids)
            #print(valid_edge_vecs)
            static_proj = self.static_proj(valid_edge_static)  

            final_edge_vecs = valid_edge_vecs + static_proj  

            seeker_vec = agent_features[i]  
            seeker_vec = tf.expand_dims(seeker_vec, axis=0)  

            # Dot: (1, embed_dim) * (num_valid, embed_dim) → (num_valid,)
            scores = tf.reduce_sum(seeker_vec * final_edge_vecs, axis=-1)
            #print(scores)
            probs = tf.nn.softmax(scores) 
            #print(probs)
            all_probs.append(probs)

        return all_probs


In [7]:
nodes = pd.read_csv(r"C:\Users\asriv\Project Amber\DSMMetro_nodes.csv")
nodes.set_index('Node', inplace=True)
nodes = nodes.drop('Unnamed: 0', axis=1)

#df = pd.read_parquet(r"C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\replays")
df = pd.read_parquet(r"C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\replays\21015323_replay_91.parquet")
df['learned_policy'] = df['learned_policy'].apply(json.loads)
edges = pd.read_csv(r'C:\Users\asriv\source\repos\Nemesis-Engine\Nemesis-Engine\data\DSMMetro_edges.csv')

EdgeIDHash = BuildEdgeIDHash(edges)
NodeToEdgesHash = BuildNodeToEdgesHash(edges)
staticFeatures = BuildStaticFeatures(edges)

In [11]:
len(df['observation_vectors'].values[0][0])

19

In [1]:
df['learned_policy'][0][0]

NameError: name 'df' is not defined

In [218]:
row = df["seeker_positions"].iloc[0]  # This gives you a list of 5 node IDs

valid_edge_ids_batch = build_valid_edge_ids_batch(row, NodeToEdgesHash)
valid_edge_ids_batch

[<tf.Tensor: shape=(6,), dtype=int32, numpy=array([ 3922,  3923,  3924, 21247, 21831, 21961], dtype=int32)>,
 <tf.Tensor: shape=(6,), dtype=int32, numpy=array([14814, 14815, 14816, 32118, 32125, 32137], dtype=int32)>,
 <tf.Tensor: shape=(4,), dtype=int32, numpy=array([ 9104,  9105, 23585, 26408], dtype=int32)>,
 <tf.Tensor: shape=(6,), dtype=int32, numpy=array([ 6820,  6821,  6822, 23197, 24237, 24601], dtype=int32)>,
 <tf.Tensor: shape=(8,), dtype=int32, numpy=
 array([15606, 15607, 15608, 15609, 33188, 34413, 34421, 34560],
       dtype=int32)>]

In [147]:
valid_edge_ids_batch[0].numpy().tolist()

[11970, 11971, 27445, 29277]

[<tf.Tensor: shape=(4,), dtype=int32, numpy=array([12355, 12356, 27444, 29279], dtype=int32)>,
 <tf.Tensor: shape=(6,), dtype=int32, numpy=array([ 5989,  5990,  5991, 23292, 23371, 23416], dtype=int32)>,
 <tf.Tensor: shape=(4,), dtype=int32, numpy=array([12148, 12149, 29457, 29461], dtype=int32)>,
 <tf.Tensor: shape=(6,), dtype=int32, numpy=array([ 7066,  7067,  7068, 24321, 24433, 25683], dtype=int32)>,
 <tf.Tensor: shape=(4,), dtype=int32, numpy=array([ 4286,  4287, 21478, 21581], dtype=int32)>]

In [18]:
NodeToEdgesHash

defaultdict(list,
            {'IA158939275': [0, 1, 2, 17329, 17342, 17348],
             'IA158943282': [3, 4, 5, 17324, 17330, 17339],
             'IA158948478': [6, 7, 17314, 17336],
             'IA158948866': [8, 9, 10, 11, 17311, 17328, 17354, 17411],
             'IA158948924': [12, 13, 14, 17359, 17385, 17399],
             'IA158949077': [15, 16, 17, 17323, 17350, 17367],
             'IA158949125': [18, 19, 20, 17308, 17322, 20018],
             'IA158949790': [21, 22, 23, 17316, 17335, 17593],
             'IA158950011': [24, 25, 26, 17306, 17310, 17337],
             'IA158950243': [27, 28, 29, 17361, 17592, 17594],
             'IA158950598': [30, 31, 32, 17312, 17326, 17331],
             'IA158950741': [33, 34, 35, 17309, 20006, 20019],
             'IA158951452': [36, 37, 38, 17305, 17363, 17609],
             'IA158951484': [39, 40, 41, 42, 17600, 17603, 17606, 17608],
             'IA158953230': [43, 44, 45, 17307, 17320, 17366],
             'IA158953308': [46, 47,

In [28]:
len(df['learned_policy'].values[0][4])

354

In [ ]:
def train_policy_header(
    policy_model,
    local_obs_batch,
    valid_edge_ids_batch,
    edge_static_features,
    target_mcts_distributions,
    optimizer=None
):
    if optimizer is None:
        optimizer = Adam(learning_rate=1e-4)

    with tf.GradientTape() as tape:
        predicted_probs = policy_model(local_obs_batch, valid_edge_ids_batch, edge_static_features)

        total_loss = 0.0
        for i in range(5):
            target_dist = target_mcts_distributions[i]  # shape: [num_valid_edges_i]
            pred_dist = predicted_probs[i]              # shape: [num_valid_edges_i]
            
            loss = tf.keras.losses.KLDivergence()(target_dist, pred_dist)
            total_loss += loss

        total_loss /= 5.0  # average over agents

    grads = tape.gradient(total_loss, policy_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, policy_model.trainable_variables))
    return total_loss.numpy()